## 🥈 Silver Layer

In the Silver Layer, we will perform data cleaning and transformation to prepare the data for analysis. This includes handling missing values, converting data types, and creating new features as needed.

## Change the path to import modules from the parent directory

In [ ]:
import sys
sys.path.append("..")

## Importing necessary libraries

In [ ]:
from pyspark.sql.functions import *
from spark_session import get_spark
from pyspark.sql.types import *
from config import get_config
import pandas as pd

## Detect environment

In [ ]:
cfg = get_config()

## Change settings to display all columns in the DataFrame

In [ ]:
pd.set_option("display.max_columns", None)

## Intialize Spark session

In [ ]:
spark = get_spark("analytics-spotify-user-transform")

## Read bronze table

In [ ]:
if cfg["environment"] == "Local":
    print("✅ Environment: Local")
    
    # Read JSON files into Spark DataFrame
    df_history = pd.read_csv("../data/export/bronze/streaming_history.csv")
    df_history = spark.createDataFrame(df_history)

elif cfg["environment"] == "Databricks":
    print("✅ Environment: Databricks")
    
    # Read bronze table
    df_history = spark.table(cfg["name_table_bronze"])

## Remove unnecessary columns to create the silver layer dataframe

In [ ]:
# List of columns to drop from the bronze layer that are not needed for analysis
drop_columns = [
    "audiobook_chapter_title",
    "audiobook_chapter_uri",
    "audiobook_title",
    "audiobook_uri",
    "conn_country",
    "incognito_mode",
    "ip_addr",
    "dt_ingestion",
    "source_file"
]

# Remove unnecessary columns to create the silver layer dataframe
df_transform_silver = df_history.drop(*drop_columns)

## Function to convert timestamp to Brazil timezone

In [ ]:
def convert_to_brazil_timezone(ts_col):
    return from_utc_timestamp(ts_col, "America/Sao_Paulo")

## Rename columns to follow a consistent naming convention

In [ ]:
df_silver = (
    df_transform_silver
    .withColumnRenamed("episode_name","nm_episode_name")
    .withColumnRenamed("episode_show_name","nm_episode_show_name")
    .withColumnRenamed("master_metadata_album_album_name","nm_album_name")
    .withColumnRenamed("master_metadata_album_artist_name","nm_artist_name")
    .withColumnRenamed("master_metadata_track_name","nm_track_name")
    .withColumnRenamed("ms_played","qt_played_ms")
    .withColumnRenamed("offline","fl_offline")
    .withColumnRenamed("offline_timestamp","ts_offline")
    .withColumnRenamed("platform","ds_platform")
    .withColumnRenamed("reason_end","ds_reason_end")
    .withColumnRenamed("reason_start","ds_reason_start")
    .withColumnRenamed("shuffle","fl_shuffle")
    .withColumnRenamed("skipped","fl_skipped")
    .withColumnRenamed("spotify_episode_uri","ds_spotify_episode_uri")
    .withColumnRenamed("spotify_track_uri","ds_spotify_track_uri")
    .withColumn("ts_streaming",  col("ts").cast("Timestamp"))
    .withColumn("dt_reference", to_date(col("ts")))
)

## Create new column for the hour of the day in Brazil timezone

In [ ]:
df_silver = (
    df_silver
    .withColumn("nr_hour_brazil",convert_to_brazil_timezone(col("ts")))
)

## Create columns to calculate how many seconds and minutes the user listened to a track or episode

In [ ]:
df_silver = (
    df_silver
    .withColumn("ts_duration_seconds",(col("qt_played_ms") / 1000).cast("Int"))

    .withColumn("ts_duration_minutes",(col("qt_played_ms") / 1000 / 60).cast("Int"))

)

## Create a column for period of the day (morning, afternoon, evening, night) based on the hour in Brazil timezone

In [ ]:
df_silver = (
    df_silver
        .withColumn("ds_period_day",
            when(hour(col("nr_hour_brazil")) < 6, lit("Dawn"))
            .when(hour(col("nr_hour_brazil")) < 12, lit("Morning"))
            .when(hour(col("nr_hour_brazil")) < 18, lit("Afternoon"))
            .otherwise(lit("Evening")))
        
        .withColumn("nr_order_period_day",
            when(hour(col("nr_hour_brazil")) < 6, lit(1))
            .when(hour(col("nr_hour_brazil")) < 12, lit(2))
            .when(hour(col("nr_hour_brazil")) < 18, lit(3))
            .otherwise(lit(4)))
)


## Create new columns for reason start and device type

In [ ]:
df_silver = (
    df_silver
    .withColumn("ds_reason_start_desc", 
                when(col("ds_reason_start") == "trackdone","Track Done")
                .when(col("ds_reason_start") == "clickrow","Manual Selection")
                .when(col("ds_reason_start") == "appload","App Load")
                .when(col("ds_reason_start") == "playbtn","Play Button")
                .when(col("ds_reason_start").isin("fwdbtn", "backbtn"), "Forward/Backward Button")
                .when(col("ds_reason_start") == "remote","Remote Control")
                .otherwise("Other Reasons"))
                                  
    .withColumn("ds_device_type", 
                when(lower(col("ds_platform")).contains("android"),"Android")
                .when(lower(col("ds_platform")).contains("ios"),"iOS")
                .when(lower(col("ds_platform")).contains("web"),"Web")
                .when(lower(col("ds_platform")).contains("windows"),"Windows")
                .when(lower(col("ds_platform")).contains("mac"),"Mac")
                .when(lower(col("ds_platform")).contains("linux"),"Linux")
                .when(lower(col("ds_platform")).contains("tv"),"TV")
                .when(lower(col("ds_platform")).contains("echo_show_5"),"Echo_Show")
                .when(lower(col("ds_platform")).contains("other"),"Other")
                .otherwise("Not Identified"))
)
 

## Create a new column for link to the track, album or podcast episode

In [ ]:
df_silver = (
    df_silver
    .withColumn("ds_link_musica", 
                when(
                    col("nm_track_name").isNotNull(),
                    concat(
                        lit("https://open.spotify.com/track/"),  
                         col("ds_spotify_track_uri")
                         )
                    )
               .when(
                   col("nm_album_name").isNotNull(),
                   concat(
                       lit("https://open.spotify.com/album/"),
                       col("ds_spotify_track_uri"))
                       )
               .when(
                   col("nm_episode_show_name").isNotNull(),
                   concat(
                       lit("https://open.spotify.com/episode/"),
                       col("ds_spotify_track_uri")
                       )
                   )
               .otherwise(lit("Não identificado")))
)
 

## Create a new column for union track name and podcast name

In [ ]:
df_silver = (
    df_silver
    .withColumn(
        "nm_audio_name",
        when(
            col("nm_track_name").isNull(),
            col("nm_episode_name")
        ).otherwise(col("nm_track_name"))
    )
)

## Create a new column for union artist name and podcast show name

In [ ]:
df_silver = (
    df_silver
    .withColumn(
        "nm_owner_name",
        when(
            col("nm_artist_name").isNull(),
            col("nm_episode_show_name")
        ).otherwise(col("nm_artist_name"))
    )
)

## Create a new column with type of content (Music or Podcast)

In [ ]:
df_silver = (
    df_silver
    .withColumn(
        "audio_type",
        when(
            col("nm_track_name").isNull(),
            lit("Podcast")
        ).otherwise(lit("Music"))
    )
)

## Write the silver layer dataframe to a new table in the silver layer

In [ ]:
if cfg["environment"] == "Local":
    print("✅ Environment: Local")
    
    # Export dataframe to CSV
    df_silver.toPandas().to_csv("../data/export/silver/streaming_history.csv", index=False)
    print("✅ Exported CSV to ../data/export/silver/streaming_history.csv")

elif cfg["environment"] == "Databricks":
    print("✅ Environment: Databricks")

    # Write to Delta Lake table
    df_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(cfg["name_table_silver"])
    print(f"✅ Loaded table: {cfg['name_table_silver']}")

In [ ]:
spark.stop()